In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.dates as mdates

In [ ]:
eph = pd.read_parquet("data/panel_eph.parquet")
eph.info()

In [ ]:
# Mostrar el conteo de los valores de la variable RAMA por eph.fecha
eph.caes_seccion_label.value_counts().sort_index()

In [ ]:
# Drop observations where ANO4 is 1901.0 or 1902.0 from eph DataFrame
eph = eph[~eph.ANO4.isin([1901.0, 1902.0])]
eph.ANO4.value_counts().sort_index()

## Recreación del índice de desempleo

In [ ]:
df_empleo = eph.copy()

df_empleo["ocupado"] = (df_empleo["ESTADO"] == "Ocupado").astype(float)
df_empleo["desocupado"] = (df_empleo["ESTADO"] == "Desocupado").astype(float)
df_empleo["desocupado_ponderado"] = df_empleo["desocupado"] * df_empleo["PONDERA"]
df_empleo["poblacion_activa_ponderada"] = (df_empleo["ocupado"] + df_empleo["desocupado"]) * df_empleo["PONDERA"]

df_empleo_agg = (
    df_empleo
    .groupby(["fecha"], as_index=False)
    .agg({"desocupado_ponderado": "sum", "poblacion_activa_ponderada": "sum"})
    .rename(columns={"desocupado_ponderado": "desocupado_total", "poblacion_activa_ponderada": "poblacion_activa_total"})
    .assign(tasa_desempleo=lambda t: np.round(t["desocupado_total"] / t["poblacion_activa_total"]*100, 2))
)

df_empleo_agg = df_empleo_agg.sort_values(["fecha"])

fig, ax = plt.subplots(figsize=(14, 6))

sns.lineplot(
    data=df_empleo_agg,
    x="fecha",
    y="tasa_desempleo",
    markers=True,
    dashes=False,
    ax=ax
)

locator = mdates.AutoDateLocator()
ax.xaxis.set_major_locator(locator)
ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(locator))

# Alternativa más simple si DateFormatter("%q") no está disponible en tu versión:
# ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))

ax.set_title("Evolución de la Tasa de desempleo")
ax.set_ylabel("Tasa de desempleo ponderada")
ax.set_xlabel("Año")
ax.grid(axis="y", linestyle="--", alpha=0.35)
plt.tight_layout()
plt.show()

In [ ]:
df_empleo_agg[["fecha", "tasa_desempleo"]].to_clipboard(index=False)